In [ ]:
pip install torch --index-url https://download.pytorch.org/whl/cpu

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------------------
# Vorgaben
prompt   = "Die Bayerische Staatsoper ist eine"
model_id = "dbmdz/german-gpt2"          # für Deutsch z. B. "dbmdz/german-gpt2"
top_k    = 10               # wie viele Kandidaten anzeigen
# -------------------------------------------

# Tokenizer und Modell laden
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id)
model.eval()                        # nur Inferenz

# Prompt in Tokens verwandeln
input_ids = tokenizer(prompt, return_tensors="pt").input_ids  # Tensor shape (1, seq_len)

# Inferenz ausführen (ohne Gradienten, spart Speicher)
with torch.no_grad():
    logits = model(input_ids).logits        # shape (1, seq_len, vocab_size)

# Logits des letzten Tokens extrahieren
next_token_logits = logits[0, -1, :]        # shape (vocab_size,)

# In Wahrscheinlichkeiten umwandeln und Top-k auswählen
probs             = torch.softmax(next_token_logits, dim=-1)
values, indices   = torch.topk(probs, top_k)  # jeweils Länge k

# Ergebnisliste ausgeben
print(f"Top-{top_k}-Kandidaten für das nächste Token nach:\n»{prompt}«\n")
for rank, (idx, p) in enumerate(zip(indices.tolist(), values.tolist()), start=1):
    token = tokenizer.decode([idx]).strip()
    print(f"{rank}. '{token}'  –  p = {p:.4f}")

# Optional: Prompt mit dem besten Token (greedy) vervollständigen
generated_ids = model.generate(
    input_ids,
    max_new_tokens=1,     # genau ein Token anhängen
    do_sample=False       # greedy decoding
)
print("\nVervollständigte Ausgabe:")
print(tokenizer.decode(generated_ids[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Top-10-Kandidaten für das nächste Token nach:
»Die Bayerische Staatsoper ist eine«

1. 'der'  –  p = 0.3192
2. 'staatlich'  –  p = 0.0493
3. 'von'  –  p = 0.0413
4. 'deutsche'  –  p = 0.0331
5. 'Institution'  –  p = 0.0289
6. 'Oper'  –  p = 0.0252
7. 'traditions'  –  p = 0.0217
8. 'führende'  –  p = 0.0211
9. 'renommierte'  –  p = 0.0206
10. 'Opern'  –  p = 0.0144

Vervollständigte Ausgabe:
Die Bayerische Staatsoper ist eine der
